# Checking python version and libs for compatiablity


In [2]:
import sys
import cv2
import mediapipe as mp
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print(f"Python version: {sys.version}")
print(f"OpenCV version: {cv2.__version__}")
print(f"MediaPipe version: {mp.__version__}")
print(f"Pandas version: {pd.__version__}")
print(f"Numpy version: {np.__version__}")

mp_pose = mp.solutions.pose
print("solutions API works!")

Python version: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
OpenCV version: 4.13.0
MediaPipe version: 0.10.14
Pandas version: 3.0.3
Numpy version: 2.4.6
solutions API works!


Analyzing one frame using MediaPipe

In [3]:
# Start with the forehand video
video_path = "training_vid/forehand.mp4"

cap = cv2.VideoCapture(video_path)

print(f"Video opened: {cap.isOpened()}")
print(f"Total frames: {int(cap.get(cv2.CAP_PROP_FRAME_COUNT))}")
print(f"FPS: {cap.get(cv2.CAP_PROP_FPS)}")
print(f"Width: {int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))}")
print(f"Height: {int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))}")

cap.release()

Video opened: True
Total frames: 2131
FPS: 30.0
Width: 1920
Height: 1080


# Extracting each frame from the video to extract landmark locations in each frame

In [ ]:

def video_landmarks(video_path: str, output_csv: str):
    all_landmarks = []

    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = 0

    with mp_pose.Pose() as pose:
        while cap.isOpened():   #if vid was opened successfully
            success, frame = cap.read()     #read() returns 2 vals: bool if frame can be opened and the frame data
            
            if not success: #runs out of frames to process
                break
            
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB) #converts color from BGR to RGB because mediapipe needs this format
            results = pose.process(frame_rgb)   # processes the frame for 33 landmarks
            
            if results.pose_landmarks:
                row = {"frame": frame_count, "timestamp": frame_count/fps} #adds these 2 cols into each row
                
                for landmark in mp_pose.PoseLandmark: #records the x,y,z pos and visiility of each landmark
                    lm = results.pose_landmarks.landmark[landmark]
                    
                    #adds 4 coords of each landmark in row
                    row[f"{landmark.name}_x"] = lm.x
                    row[f"{landmark.name}_y"] = lm.y
                    row[f"{landmark.name}_z"] = lm.z
                    row[f"{landmark.name}_vis"] = lm.visibility
                
                all_landmarks.append(row)
                
            frame_count += 1
            
            if frame_count % 100 == 0:
                print(f"Processed {frame_count} frames")

    cap.release()
    df = pd.DataFrame(all_landmarks) 
    df.to_csv(output_csv, index= False)
    print(f"Done! Processed {frame_count} total frames")
    print(f"Frames with landmarks: {len(all_landmarks)}")
    print(f"Saved to {output_csv}")
    return df
                

# Saving frame data into CSV

In [6]:
video_landmarks("forehand_train.mp4")
print(f"Shape {df.shape}")
df.columns.tolist()
df[["frame", "timestamp", "RIGHT_WRIST_x", "RIGHT_WRIST_y", "LEFT_WRIST_x", "LEFT_WRIST_y"]].head(10)

Done! Processed 0 total frames
Frames with landmarks: 0


NameError: name 'df' is not defined

## Graphing wrist positions 

In [ ]:
#eliminates landmarks for the player on the left side of table, im on the right side
df_filtered = df[df["RIGHT_WRIST_x"] > .5] 

plt.scatter(df_filtered["RIGHT_WRIST_x"], df_filtered["RIGHT_WRIST_y"], alpha = .1)
plt.title("Right wrist position across all frames")
plt.xlabel("X coord")
plt.ylabel("Y coord")
plt.gca().invert_yaxis() # inverts y axis because 0 is at top in image coords
plt.show()

In [ ]:
df_filtered.to_csv("forehand_landmarks.csv", index = False) #doesn't number the rows
print("Saved to csv file")
print(f"Shape {df_filtered.shape}")
